In [43]:
import hssm
import hssm.plotting
import numpy as np
import pandas as pd
import arviz as az
import seaborn as sns
from matplotlib import pyplot as plt
import bambi as bmb
import statsmodels.api as sm
from pymer4.models import Lmer
from scipy import stats
from numpy.polynomial.legendre import legvander
from pathlib import Path
from numpy.polynomial.polynomial import polyvander


In [ ]:
# requirements.txt provides the environment information you need for running this script
 ## Run the following in your terminal to setup the required python environment 
 ### conda create -n myenv python=3.10
 ### conda activate myenv
 ### pip install -r requirements.txt

# analysis.ipynb is the main analysis script.

# This script is used to generate the output for different models. 

In [45]:
base = Path("../../model_output")

(base / "study_1").mkdir(parents=True, exist_ok=True)
(base / "study_2").mkdir(parents=True, exist_ok=True)

In [ ]:
# STUDY 1 DDM Model

In [31]:

pair_df_study1 = pd.read_csv('../task_data/study_1/pairings.csv')

In [33]:
pair_df_study1['rt'] = pair_df_study1['rt']/1000
pair_df_study1.loc[pair_df_study1['response']==0,'response'] = -1

In [34]:
# define conditions
conditions = [
    (pair_df_study1["valence_left"].eq("positive") & pair_df_study1["valence_right"].eq("negative")),  # left pos, right neg
    (pair_df_study1["valence_left"].eq("negative") & pair_df_study1["valence_right"].eq("positive"))   # left neg, right pos
]

# define choices
choices = [-1, 1]

# assign values
pair_df_study1["delta_valence"] = np.select(conditions, choices, default=0)

In [ ]:
# Full Model with self-belief and valence regressors

model1 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence + (1 + r_diff_norm + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples1 = model1.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples1.to_netcdf('../../model_output/study_1/drift_full_model.nc')
print("Over")

In [ ]:
# Depression Model

model2 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence*group_int + (1 + r_diff_norm + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples2 = model2.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples2.to_netcdf('../../model_output/study_1/drift_depression_model.nc')
print("Over")

In [ ]:
# Self-Belief only model

model3 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + (1 + r_diff_norm|subject)",       
        }
    ],
    
)

ddm_samples3 = model3.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples3.to_netcdf('../../model_output/study_1/drift_self_belief_model.nc')
print("Over")

In [ ]:
# Valence only model

model4 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + delta_valence + (1 + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples4 = model4.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples4.to_netcdf('../../model_output/study_1/drift_valence_model.nc')
print("Over")

In [ ]:
# No regressors model

model5 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + (1 |subject)",       
        }
    ],
    
)

ddm_samples5 = model5.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples5.to_netcdf('../../model_output/study_1/drift_no_reg_model.nc')
print("Over")

In [ ]:
# Threshold Depression model 

model6 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + (1 + r_diff_norm|subject)",       
        },
        {
            "name": "a",
            "formula": "a ~ 1 + delta_valence*group_int + (1 + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples6 = model6.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples6.to_netcdf('../../model_output/study_1/threshold_depression_model.nc')
print("Over")


In [ ]:
# STUDY 2 DDM Model

In [35]:

pair_df_study2 = pd.read_csv('../task_data/study_2/pairings.csv')


In [38]:
pair_df_study2['rt'] = pair_df_study2['rt']/1000
pair_df_study2['phq_scale'] = pair_df_study2['phq_score']/27

In [39]:
# define conditions
conditions = [
    (pair_df_study2["valence_left"].eq("positive") & pair_df_study2["valence_right"].eq("negative")),  # left pos, right neg
    (pair_df_study2["valence_left"].eq("negative") & pair_df_study2["valence_right"].eq("positive"))   # left neg, right pos
]

# define choices
choices = [-1, 1]

# assign values
pair_df_study2["delta_valence"] = np.select(conditions, choices, default=0)

In [ ]:
# Full Model with self-belief and valence regressors

model7 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence + (1 + r_diff_norm + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples7 = model7.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples7.to_netcdf('../../model_output/study_2/drift_full_model.nc')
print("Over")

In [ ]:
# Depression Model

model8 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence*phq_scale + (1 + r_diff_norm + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples8 = model8.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples8.to_netcdf('../../model_output/study_2/drift_depression_model.nc')
print("Over")

In [ ]:
# Self-Belief only model

model9 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + (1 + r_diff_norm|subject)",       
        }
    ],
    
)

ddm_samples9 = model9.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples9.to_netcdf('../../model_output/study_2/drift_self_belief_model.nc')
print("Over")

In [ ]:
# Valence only model

model10 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + delta_valence + (1 + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples10 = model10.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples10.to_netcdf('../../model_output/study_2/drift_valence_model.nc')
print("Over")

In [ ]:
# No regressors model

model11 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + (1 |subject)",       
        }
    ],
    
)

ddm_samples11 = model11.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples11.to_netcdf('../../model_output/study_2/drift_no_reg_model.nc')
print("Over")

In [ ]:
# Threshold Depression model 

model12 = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + (1 + r_diff_norm|subject)",       
        },
        {
            "name": "a",
            "formula": "a ~ 1 + delta_valence*phq_scale + (1 + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples12 = model12.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples12.to_netcdf('../../model_output/study_2/threshold_depression_model.nc')
print("Over")


In [ ]:
# Robustness Check - Drift Diffusion Models - Study 1


In [ ]:

# DDM with random intercepts on threshold (a) and starting point (z) (Supplementary Table 28)

model50b = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence + (1 + r_diff_norm + delta_valence|subject)",       
        },
        {
            "name": "a",
            "formula": "a ~ 1 + (1|subject)",       
        },
        {
            "name": "z",
            "formula": "z ~ 1 + (1|subject)",       
        }
    ],
    
)

ddm_samples50b = model50b.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples50b.to_netcdf('../../model_output/study_1/ddm_model50b.nc')
print("Over")


In [ ]:
# DDM with random intercepts on threshold (a) and starting point (z) with Depression Interaction (Supplementary Table 29)

model51b = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence*group_int + (1 + r_diff_norm + delta_valence|subject)",       
        },
        {
            "name": "a",
            "formula": "a ~ 1 + (1|subject)",       
        },
        {
            "name": "z",
            "formula": "z ~ 1 + (1|subject)",       
        }
    ],
    
)

ddm_samples51b = model51b.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples51b.to_netcdf('../../model_output/study_1/ddm_model51b.nc')
print("Over")

In [ ]:
# DDM with depression regressor on non decision time (Supplementary Table 30)

model40b = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence*group_int + (1 + r_diff_norm + delta_valence|subject)",       
        },
        {
            "name": "t",
            "formula": "t ~ 1 + group_int",       
        }
    ],
    
)

ddm_samples40b = model40b.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples40b.to_netcdf('../../model_output/study_1/ddm_model40b.nc')
print("Over")

In [ ]:
# Robustness Check - Drift Diffusion Models -  Study 2


In [ ]:

# DDM with random intercepts on threshold (a) and starting point (z) (Supplementary Table 28)

model50b = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence + (1 + r_diff_norm + delta_valence|subject)",       
        },
        {
            "name": "a",
            "formula": "a ~ 1 + (1|subject)",       
        },
        {
            "name": "z",
            "formula": "z ~ 1 + (1|subject)",       
        }
    ],
    
)

ddm_samples50b = model50b.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 2,target_accept=0.95)
ddm_samples50b.to_netcdf('../../model_output/study_2/ddm_model50b_study2.nc')
print("Over")


In [ ]:
# DDM with random intercepts on threshold (a) and starting point (z) with Depression Interaction (Supplementary Table 29)

model51b = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence*phq_scale + (1 + r_diff_norm + delta_valence|subject)",       
        },
        {
            "name": "a",
            "formula": "a ~ 1 + (1|subject)",       
        },
        {
            "name": "z",
            "formula": "z ~ 1 + (1|subject)",       
        }
    ],
    
)

ddm_samples51b = model51b.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples51b.to_netcdf('../../model_output/study_2/ddm_model51b_study2.nc')
print("Over")

In [ ]:
# DDM with depression regressor on non decision time (Supplementary Table 30)

model40b = hssm.HSSM(
    model = 'ddm',
    data = pair_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence*phq_scale + (1 + r_diff_norm + delta_valence|subject)",       
        },
        {
            "name": "t",
            "formula": "t ~ 1 + phq_scale",       
        }
    ],
    
)

ddm_samples40b = model40b.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples40b.to_netcdf('../../model_output/study_2/ddm_model40b_study2.nc')
print("Over")

In [ ]:
# Recall Model

# Generating Figure 7

recall_study1 = pd.read_csv('../task_data/study_1/recall.csv')
recall_study2 = pd.read_csv('../task_data/study_2/recall.csv')

phq_study2 = pd.read_csv('../task_data/study_2/phq.csv')
phq_study2['group'] = pd.qcut(phq_study2["phq_score"], q=3, labels=["low", "medium", "high"])

recall_study1["group"] = np.where(recall_study1["phq_score"] > 14,1,-1)
recall_study2 = pd.merge(recall_study2,phq_study2, on = ['subject','phq_score'], how = 'inner')

recall_study1['r_scale'] = recall_study1['response']/100
recall_study2['r_scale'] = recall_study2['response']/100
recall_study2['phq_scale'] = recall_study2['phq_score']/27


In [ ]:
#Study 1

no_ratings_model = bmb.Model(
    "recalled ~ valence*C(group) + count_app + (valence + count_app|subject)", data=recall_study1, family="bernoulli")

no_ratings_results = no_ratings_model.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=1,
    target_accept=0.95
)

az.to_netcdf(no_ratings_results,'../../model_output/study_1/no_ratings_recall.nc')

with_ratings_model = bmb.Model(
    "recalled ~ r_scale + valence*C(group) + count_app + (r_scale + valence + count_app|subject)", data=recall_study1, family="bernoulli")

with_ratings_results = with_ratings_model.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=1,
    target_accept=0.95
)

az.to_netcdf(with_ratings_results,'../../model_output/study_1/with_ratings_recall.nc')


In [ ]:
# Study 2
# Bayesian Analysis

no_ratings_model = bmb.Model(
    "recalled ~ valence*phq_scale + count_app + (valence + count_app|subject)", data=recall_study2, family="bernoulli")

no_ratings_results = no_ratings_model.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=1,
    target_accept=0.95
)
az.to_netcdf(no_ratings_results,'../../model_output/study_2/no_ratings_recall.nc')


with_ratings_model = bmb.Model(
    "recalled ~ r_scale + valence*phq_scale + count_app + (r_scale + valence + count_app|subject)", data=recall_study2, family="bernoulli")

with_ratings_results = with_ratings_model.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=1,
    target_accept=0.95
)
az.to_netcdf(with_ratings_results,'../../model_output/study_2/with_ratings_recall.nc')



In [5]:
#Bayesian Regression (for Choice and RT coefficients)

# STUDY 1

pair_df_study1['r_diff_norm_abs'] = pair_df_study1['r_diff_norm'].abs()
pair_df_study1["rt_z"] = pair_df_study1.groupby("subject")["rt"].transform(
    lambda x: (x - x.mean()) / x.std()
)

In [ ]:
model_choice = bmb.Model(
    "choice ~ delta_valence + r_diff_norm + "
    "(1 + delta_valence + r_diff_norm| subject)",
    data=pair_df_study1,
    family="bernoulli"
)

results_choice = model_choice.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.95
)

az.to_netcdf(results_choice, '../../model_output/study_1/choice_regressor_model.nc')

In [6]:
# Robustness Check - Delta Self-Belief not rescaled at subject level

pair_df_study1['d_sb_raw'] = pair_df_study1['rating_right'] - pair_df_study1['rating_left']

pair_df_study1["d_sb_raw_mod"] = (
(pair_df_study1["d_sb_raw"] - pair_df_study1["d_sb_raw"].min()) / (pair_df_study1["d_sb_raw"].max() - pair_df_study1["d_sb_raw"].min())
)

pair_df_study1['d_sb_raw_mod'] = pair_df_study1['d_sb_raw_mod'] - 0.5


In [ ]:
model_choice_rc = bmb.Model(
    "choice ~ delta_valence + d_sb_raw_mod + "
    "(1 + delta_valence + d_sb_raw_mod| subject)",
    data=pair_df_study1,
    family="bernoulli"
)

results_choice_rc = model_choice_rc.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.95
)

az.to_netcdf(results_choice_rc, '../../model_output/study_1/choice_regressor_model_rob_check.nc')

In [ ]:
model_rt = bmb.Model(
    "rt_z ~ val_left + val_right + r_diff_norm_abs + "
    "(1 + val_left + val_right + r_diff_norm_abs | subject)",
    data=pair_df_study1,
    family="gaussian"
)

results_rt = model_rt.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.95
)

az.to_netcdf(results_rt, '../../model_output/study_1/rt_regressor_model.nc')

In [ ]:
# Robustness Check - Delta Self-Belief not rescaled at subject level,  RT not z scored at subject level

pair_df_study1["rt_z2"] = (
    pair_df_study1["rt"] - pair_df_study1["rt"].mean()
    ) / pair_df_study1["rt"].std(ddof=1) 


pair_df_study1["d_sb_raw_mod_abs"] = pair_df_study1["d_sb_raw_mod"].abs()

model_rt_rc = bmb.Model(
    "rt_z2 ~ val_left + val_right + d_sb_raw_mod_abs + "
    "(1 + val_left + val_right + d_sb_raw_mod_abs | subject)",
    data=pair_df_study1,
    family="gaussian"
)

results_rt_rc = model_rt_rc.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.95
)

az.to_netcdf(results_rt_rc, '../../model_output/study_1/rt_regressor_model_rob_check.nc')

In [ ]:
# RT:Valence regressor

pair_pos_neg = pair_df_study1[pair_df_study1['valence']=='PosNeg'].copy()

pair_pos_neg['positive_rating'] = np.where(
        pair_pos_neg['valence_left'] == 'positive',
        pair_pos_neg['rating_left'],
        pair_pos_neg['rating_right']
    )

pair_pos_neg['negative_rating'] = np.where(
        pair_pos_neg['valence_left'] == 'negative',
        pair_pos_neg['rating_left'],
        pair_pos_neg['rating_right']
    )

pair_pos_neg['delta_pos_rat'] = pair_pos_neg['positive_rating'] - pair_pos_neg['negative_rating']

pair_pos_neg["pos_neg_z"] = pair_pos_neg.groupby("subject")["delta_pos_rat"].transform(
    lambda x: (x - x.mean()) / x.std(ddof=1))  # ddof=0 = population SD, use ddof=1 for sample SD


# Ensure centered / scaled predictor (important!)
x = pair_pos_neg["pos_neg_z"].to_numpy()

Q, _ = np.linalg.qr(polyvander(x, 2))
pos_neg_lin  = Q[:, 1]
pos_neg_quad = Q[:, 2]

pair_pos_neg["pos_neg_lin"] = pos_neg_lin
pair_pos_neg["pos_neg_quad"] = pos_neg_quad

model_rt = bmb.Model(
        "rt_z ~ pos_neg_lin + pos_neg_quad + "
        "(1 + pos_neg_lin + pos_neg_quad| subject)",
        data=pair_pos_neg,
        family="gaussian"
    )

results_rt2 = model_rt.fit(
        draws=5000,
        tune=2000,
        chains=4,
        cores=1,
        target_accept=0.95
    )

az.to_netcdf(results_rt2, '../../model_output/study_1/rt_valence_regressor_model.nc')

In [ ]:
# Robustness Check 
# Pos-Neg not z-scored at subject level
# RT not z-scored at subject level
# RT regressors not orthogonalized

pair_pos_neg["pos_neg_z2"] = (
    pair_pos_neg["delta_pos_rat"] - pair_pos_neg["delta_pos_rat"].mean()
    ) / pair_pos_neg["delta_pos_rat"].std(ddof=1) 

pair_pos_neg["pos_neg_z2_squared"] = pair_pos_neg["pos_neg_z2"] ** 2

pair_pos_neg["pos_neg_lin"] = pair_pos_neg["pos_neg_z2"]
pair_pos_neg["pos_neg_quad"] = pair_pos_neg["pos_neg_z2_squared"]

model_rt_rc = bmb.Model(
        "rt_z2 ~ pos_neg_lin + pos_neg_quad + "
        "(1 + pos_neg_lin + pos_neg_quad| subject)",
        data=pair_pos_neg,
        family="gaussian"
    )

results_rt2_rc = model_rt_rc.fit(
        draws=5000,
        tune=2000,
        chains=4,
        cores=1,
        target_accept=0.95
    )

az.to_netcdf(results_rt2_rc, '../../model_output/study_1/rt_valence_regressor_model_rob_check.nc')

In [ ]:
# Robustness Check 
# Pos-Neg not z-scored at subject level
# RT not z-scored at subject level
# RT regressors orthogonalized

# Ensure centered / scaled predictor (important!)
x = pair_pos_neg["pos_neg_z2"].to_numpy()

Q, _ = np.linalg.qr(polyvander(x, 2))
pos_neg_lin  = Q[:, 1]
pos_neg_quad = Q[:, 2]

pair_pos_neg["pos_neg_lin"] = pos_neg_lin
pair_pos_neg["pos_neg_quad"] = pos_neg_quad

model_rt_rc = bmb.Model(
        "rt_z2 ~ pos_neg_lin + pos_neg_quad + "
        "(1 + pos_neg_lin + pos_neg_quad| subject)",
        data=pair_pos_neg,
        family="gaussian"
    )

results_rt2_rc = model_rt_rc.fit(
        draws=5000,
        tune=2000,
        chains=4,
        cores=1,
        target_accept=0.95
    )

az.to_netcdf(results_rt2_rc, '../../model_output/study_1/rt_valence_regressor_model_rob_check_with_ortho.nc')

In [41]:
# STUDY 2

# val_left and val_right columns not present in Study 2

pair_df_study2['val_left'] = np.where(pair_df_study2['valence_left']=='negative',1,-1)
pair_df_study2['val_right'] = np.where(pair_df_study2['valence_right']=='negative',-1,1)
pair_df_study2['r_diff_norm_abs'] = pair_df_study2['r_diff_norm'].abs()
pair_df_study2["rt_z"] = pair_df_study2.groupby("subject")["rt"].transform(
    lambda x: (x - x.mean()) / x.std()
)



In [ ]:
model_choice = bmb.Model(
    "choice ~ delta_valence + r_diff_norm + "
    "(1 + delta_valence + r_diff_norm| subject)",
    data=pair_df_study2,
    family="bernoulli"
)

results_choice = model_choice.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.95
)

az.to_netcdf(results_choice, '../../model_output/study_2/choice_regressor_model.nc')

In [15]:
# Robustness Check - Delta Self-Belief not rescaled at subject level

pair_df_study2['d_sb_raw'] = pair_df_study2['rating_right'] - pair_df_study2['rating_left']

pair_df_study2["d_sb_raw_mod"] = (
(pair_df_study2["d_sb_raw"] - pair_df_study2["d_sb_raw"].min()) / (pair_df_study2["d_sb_raw"].max() - pair_df_study2["d_sb_raw"].min())
)

pair_df_study2['d_sb_raw_mod'] = pair_df_study2['d_sb_raw_mod'] - 0.5


In [ ]:
model_choice_rc = bmb.Model(
    "choice ~ delta_valence + d_sb_raw_mod + "
    "(1 + delta_valence + d_sb_raw_mod| subject)",
    data=pair_df_study2,
    family="bernoulli"
)

results_choice_rc = model_choice_rc.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.95
)

az.to_netcdf(results_choice_rc, '../../model_output/study_2/choice_regressor_model_rob_check.nc')

In [ ]:
model_rt = bmb.Model(
    "rt_z ~ val_left + val_right + r_diff_norm_abs + "
    "(1 + val_left + val_right + r_diff_norm_abs | subject)",
    data=pair_df_study2,
    family="gaussian"
)

results_rt = model_rt.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.95
)

az.to_netcdf(results_rt, '../../model_output/study_2/rt_regressor_model.nc')

In [ ]:
# Robustness Check - Delta Self-Belief not rescaled at subject level,  RT not z scored at subject level

pair_df_study2["rt_z2"] = (
    pair_df_study2["rt"] - pair_df_study2["rt"].mean()
    ) / pair_df_study2["rt"].std(ddof=1) 


pair_df_study2["d_sb_raw_mod_abs"] = pair_df_study2["d_sb_raw_mod"].abs()

model_rt_rc = bmb.Model(
    "rt_z2 ~ val_left + val_right + d_sb_raw_mod_abs + "
    "(1 + val_left + val_right + d_sb_raw_mod_abs | subject)",
    data=pair_df_study2,
    family="gaussian"
)

results_rt_rc = model_rt_rc.fit(
    draws=5000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.95
)

az.to_netcdf(results_rt_rc, '../../model_output/study_2/rt_regressor_model_rob_check.nc')

In [ ]:
# RT:Valence regressor

pair_pos_neg = pair_df_study2[pair_df_study2['valence']=='PosNeg'].copy()

pair_pos_neg['positive_rating'] = np.where(
        pair_pos_neg['valence_left'] == 'positive',
        pair_pos_neg['rating_left'],
        pair_pos_neg['rating_right']
    )

pair_pos_neg['negative_rating'] = np.where(
        pair_pos_neg['valence_left'] == 'negative',
        pair_pos_neg['rating_left'],
        pair_pos_neg['rating_right']
    )

pair_pos_neg['delta_pos_rat'] = pair_pos_neg['positive_rating'] - pair_pos_neg['negative_rating']

pair_pos_neg["pos_neg_z"] = pair_pos_neg.groupby("subject")["delta_pos_rat"].transform(
    lambda x: (x - x.mean()) / x.std(ddof=1))  # ddof=0 = population SD, use ddof=1 for sample SD


# Ensure centered / scaled predictor (important!)
x = pair_pos_neg["pos_neg_z"].to_numpy()

Q, _ = np.linalg.qr(polyvander(x, 2))
pos_neg_lin  = Q[:, 1]
pos_neg_quad = Q[:, 2]

pair_pos_neg["pos_neg_lin"] = pos_neg_lin
pair_pos_neg["pos_neg_quad"] = pos_neg_quad

model_rt = bmb.Model(
        "rt_z ~ pos_neg_lin + pos_neg_quad + "
        "(1 + pos_neg_lin + pos_neg_quad| subject)",
        data=pair_pos_neg,
        family="gaussian"
    )

results_rt2 = model_rt.fit(
        draws=5000,
        tune=2000,
        chains=4,
        cores=1,
        target_accept=0.95
    )

az.to_netcdf(results_rt2, '../../model_output/study_2/rt_valence_regressor_model.nc')

In [ ]:
# Robustness Check 
# Pos-Neg not z-scored at subject level
# RT not z-scored at subject level
# RT regressors not orthogonalized

pair_pos_neg["pos_neg_z2"] = (
    pair_pos_neg["delta_pos_rat"] - pair_pos_neg["delta_pos_rat"].mean()
    ) / pair_pos_neg["delta_pos_rat"].std(ddof=1) 

pair_pos_neg["pos_neg_z2_squared"] = pair_pos_neg["pos_neg_z2"] ** 2

pair_pos_neg["pos_neg_lin"] = pair_pos_neg["pos_neg_z2"]
pair_pos_neg["pos_neg_quad"] = pair_pos_neg["pos_neg_z2_squared"]

model_rt_rc = bmb.Model(
        "rt_z2 ~ pos_neg_lin + pos_neg_quad + "
        "(1 + pos_neg_lin + pos_neg_quad| subject)",
        data=pair_pos_neg,
        family="gaussian"
    )

results_rt2_rc = model_rt_rc.fit(
        draws=5000,
        tune=2000,
        chains=4,
        cores=4,
        target_accept=0.95
    )

az.to_netcdf(results_rt2_rc, '../../model_output/study_2/rt_valence_regressor_model_rob_check.nc')

In [ ]:
# Robustness Check 
# Pos-Neg not z-scored at subject level
# RT not z-scored at subject level
# RT regressors orthogonalized

# Ensure centered / scaled predictor (important!)
x = pair_pos_neg["pos_neg_z2"].to_numpy()

Q, _ = np.linalg.qr(polyvander(x, 2))
pos_neg_lin  = Q[:, 1]
pos_neg_quad = Q[:, 2]

pair_pos_neg["pos_neg_lin"] = pos_neg_lin
pair_pos_neg["pos_neg_quad"] = pos_neg_quad

model_rt_rc = bmb.Model(
        "rt_z2 ~ pos_neg_lin + pos_neg_quad + "
        "(1 + pos_neg_lin + pos_neg_quad| subject)",
        data=pair_pos_neg,
        family="gaussian"
    )

results_rt2_rc = model_rt_rc.fit(
        draws=5000,
        tune=2000,
        chains=4,
        cores=1,
        target_accept=0.95
    )

az.to_netcdf(results_rt2_rc, '../../model_output/study_2/rt_valence_regressor_model_rob_check_with_ortho.nc')

In [ ]:
# DIC Calculation


# Loading the DDM models

drift_full_model_study1 = az.InferenceData.from_netcdf('../../model_output/study_1/drift_full_model.nc')
drift_full_model_study2 = az.InferenceData.from_netcdf('../../model_output/study_2/drift_full_model.nc')

drift_self_belief_model_study1 = az.InferenceData.from_netcdf('../../model_output/study_1/drift_self_belief_model.nc')
drift_valence_model_study1 = az.InferenceData.from_netcdf('../../model_output/study_1/drift_valence_model.nc')
drift_no_reg_model_study1 = az.InferenceData.from_netcdf('../../model_output/study_1/drift_no_reg_model.nc')

drift_self_belief_model_study2 = az.InferenceData.from_netcdf('../../model_output/study_2/drift_self_belief_model.nc')
drift_valence_model_study2 = az.InferenceData.from_netcdf('../../model_output/study_2/drift_valence_model.nc')
drift_no_reg_model_study2 = az.InferenceData.from_netcdf('../../model_output/study_2/drift_no_reg_model.nc')


In [ ]:

# Define your models now (Model Number, Model name)
models = [
    ("self_belief", "v ~ 1 + r_diff_norm"),
    ("no_reg", "v ~ 1"),
    ("valence", "v ~ 1 + delta_valence"),
    ("full", "v ~ 1 + r_diff_norm + delta_valence"),
]

# Create the DataFrame; make DIC an empty (NaN) column for now
model_df = pd.DataFrame(models, columns=["Model Number", "Model name"])
model_df["DIC"] = np.nan

print(model_df)


In [ ]:
def _total_loglik_from_idata(idata: az.InferenceData):
    """
    Sum log-likelihood over all observation dims for each (chain, draw).
    Works even if multiple likelihood vars are present.
    """
    if not hasattr(idata, "log_likelihood") or idata.log_likelihood is None:
        raise ValueError(
            "No log_likelihood found in InferenceData. "
            "Re-fit with log_likelihood saved (e.g., PyMC: idata_kwargs={'log_likelihood': True})."
        )
    ll_group = idata.log_likelihood
    total_ll = None
    for _, da in ll_group.data_vars.items():
        dims_to_sum = [d for d in da.dims if d not in ("chain", "draw")]
        da_sum = da.sum(dim=dims_to_sum)
        total_ll = da_sum if total_ll is None else total_ll + da_sum
    return total_ll  # dims: chain, draw

def dic_from_idata(idata: az.InferenceData):
    """
    DIC with variance-based p_D:
      D_s   = -2 * loglik_total per sample
      D_bar = mean(D_s)
      p_D   = 0.5 * var(D_s)
      DIC   = D_bar + p_D
    """
    total_ll = _total_loglik_from_idata(idata)                # (chain, draw)
    D = (-2.0 * total_ll).stack(sample=("chain", "draw"))     # (sample,)
    D_vals = np.asarray(D.values)
    D_vals = D_vals[~np.isnan(D_vals)]
    ddof = 1 if D_vals.size > 1 else 0
    D_bar = float(D_vals.mean())
    p_D = float(0.5 * D_vals.var(ddof=ddof))
    DIC = D_bar + p_D
    return {"D_bar": D_bar, "p_D": p_D, "DIC": DIC}


In [ ]:
name_lookup = dict(zip(model_df["Model Number"], model_df["Model name"]))

def print_and_update_dic(model_number, idata):
    name = name_lookup.get(model_number, "<unknown>")
    try:
        out = dic_from_idata(idata)              # uses your function
        dic = out["DIC"]
        # update table
        model_df.loc[model_df["Model Number"] == model_number, "DIC"] = dic
        # print nicely
        print(f"Model {model_number} | {name} | DIC = {dic:.3f}")
    except Exception as e:
        # keep NA in table and report error
        print(f"Model {model_number} | {name} | DIC = NA ({e})")

print_and_update_dic("self_belief",drift_self_belief_model_study1)
print_and_update_dic("no_reg",drift_no_reg_model_study1)
print_and_update_dic("valence",drift_valence_model_study1)
print_and_update_dic("full",drift_full_model_study1)


In [ ]:
model_df_study1 = model_df.copy()
model_df['DIC'] = np.nan

In [ ]:
model_df_study1.to_csv("../../model_output/study_1/dic_models.csv", index=False)

In [ ]:
# Study 2 
# NOTE - This is memory intensive. Recommend using server/cluster rather than local computer. Can skip this section unless interested in this specific analysis

print_and_update_dic("self_belief",drift_self_belief_model_study2)
print_and_update_dic("no_reg",drift_no_reg_model_study2)
print_and_update_dic("valence",drift_valence_model_study2)
print_and_update_dic("full",drift_full_model_study2)

In [ ]:
model_df_study2 = model_df.copy()
model_df_study2.head()

model_df_study2.to_csv("../../model_output/study_2/dic_models.csv", index=False)

In [ ]:
# Parameter Recovery for drift_full_model

In [ ]:
# Generating the True Estimates 


def gen_true_estimates(pair_data,drift_full_model,study): 

    ################### Calculating the random effects ##################
    
    # r_diff_norm
    random_effects_v_r_diff_norm = drift_full_model.posterior["v_r_diff_norm|subject"]
    subject_means_v_r_diff_norm = random_effects_v_r_diff_norm.mean(dim=["chain", "draw"])

    subject_means_v_r_diff_norm_df = subject_means_v_r_diff_norm.to_series().reset_index()
    subject_means_v_r_diff_norm_df.columns = ['subject', 're_v_r_diff_norm']

    # v intercept
    random_effects_v_intercept = drift_full_model.posterior["v_1|subject"]
    subject_means_v_intercept = random_effects_v_intercept.mean(dim = ["chain","draw"])

    subject_means_v_intercept_df = subject_means_v_intercept.to_series().reset_index()
    subject_means_v_intercept_df.columns = ['subject', 're_v_intercept']

    # v delta_valence
    random_effects_v_delta_valence = drift_full_model.posterior["v_delta_valence|subject"]
    subject_means_v_delta_valence = random_effects_v_delta_valence.mean(dim = ["chain","draw"])

    subject_means_v_delta_valence_df = subject_means_v_delta_valence.to_series().reset_index()
    subject_means_v_delta_valence_df.columns = ['subject', 're_v_delta_valence']

    random_effects_df = pd.merge(subject_means_v_intercept_df, subject_means_v_r_diff_norm_df, on='subject', how='inner')
    random_effects_df = pd.merge(random_effects_df,subject_means_v_delta_valence_df, on='subject', how='inner')

    random_effects_df.head()

    # Fixed Effects

    fixed_effects_df = az.summary(drift_full_model, var_names = ['t','z','a','v_r_diff_norm',
                                              'v_delta_valence','v_Intercept'])

    new_fixed_effects_df = fixed_effects_df['mean'].T.to_frame().T
    new_fixed_effects_df.columns = fixed_effects_df.index

    #################### Subject Effects (Fixed + Random Effects) ########################

    subject_effects_df = random_effects_df.copy()

    subject_effects_df['re_v_r_diff_norm'] = subject_effects_df['re_v_r_diff_norm'] + new_fixed_effects_df['v_r_diff_norm'].values[0]
    subject_effects_df['re_v_intercept'] = subject_effects_df['re_v_intercept'] + new_fixed_effects_df['v_Intercept'].values[0]
    subject_effects_df['re_v_delta_valence'] = subject_effects_df['re_v_delta_valence'] + new_fixed_effects_df['v_delta_valence'].values[0]

    subject_effects_df['z'] = new_fixed_effects_df['z'].values[0]
    subject_effects_df['t'] = new_fixed_effects_df['t'].values[0]
    subject_effects_df['a'] = new_fixed_effects_df['a'].values[0]

    subject_effects_df = subject_effects_df.rename(columns = {"re_v_r_diff_norm":"v_r_diff_norm",
                                                         "re_v_delta_valence":"v_delta_valence", 
                                                         "re_v_intercept":"v_intercept"})


    pair_data = pd.merge(pair_data,subject_effects_df, on = 'subject', how = 'inner')

    #################### Simulate Data with the true estimates ##############################

    v_trial_wise = (pair_data['v_intercept'] + pair_data['v_r_diff_norm']*pair_data['r_diff_norm'] +
                    pair_data['v_delta_valence']*pair_data['delta_valence']) 

    a_est = pair_data["a"].values[0]
    z_est = pair_data["z"].values[0]
    t_est = pair_data["t"].values[0]
    v_est = v_trial_wise.values

    n_sims = 1
    
    sim_data = hssm.simulate_data(
        model="ddm",
        theta=dict(
            v=v_est,
            a=a_est,
            z=z_est,
            t=t_est,
        ),
        size=n_sims, # For parameter recovery
    )

    sim_data['subject'] = np.tile(pair_data['subject'],n_sims)
    sim_data['sim_num'] = np.repeat(np.arange(1,n_sims+1),pair_data.shape[0])
    sim_data['r_diff_norm'] = np.tile(pair_data['r_diff_norm'],n_sims)
    sim_data['valence'] = np.tile(pair_data['valence'],n_sims)
    sim_data['valence_left'] = np.tile(pair_data['valence_left'],n_sims)
    sim_data['valence_right'] = np.tile(pair_data['valence_right'],n_sims)
    sim_data['rating_left'] = np.tile(pair_data['rating_left'],n_sims)
    sim_data['rating_right'] = np.tile(pair_data['rating_right'],n_sims)
    sim_data['delta_valence'] = np.tile(pair_data['delta_valence'],n_sims)
    sim_data['v_intercept'] = np.tile(pair_data['v_intercept'],n_sims)
    sim_data['v_delta_valence'] = np.tile(pair_data['v_delta_valence'],n_sims)
    sim_data['v_r_diff_norm'] = np.tile(pair_data['v_r_diff_norm'],n_sims)

    sim_data['choice'] = np.where(sim_data['response'] == -1, 0, 1)
    sim_data['rt'] = 1000*sim_data['rt']

    sim_data.to_csv(f"../task_data/{study}/drift_full_model_pr_true_param.csv", index=False)


In [ ]:
# Generating the True Estimates 

gen_true_estimates(pair_df_study1,drift_full_model_study1,'study_1')
gen_true_estimates(pair_df_study2,drift_full_model_study2,'study_2')


In [ ]:
# Fitting the dataset simulated using the True Estimates (Study 1) 

#param_recovery_model_20a_study1.csv -> Contains the true parameter estimates

pr_df_study1 = pd.read_csv('../task_data/study_1/drift_full_model_pr_true_param.csv')
pr_df_study1['rt'] = pr_df_study1['rt']/1000

model_a = hssm.HSSM(
    model = 'ddm',
    data = pr_df_study1,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence + (1 + r_diff_norm + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples_a = model_a.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples_a.to_netcdf('../../model_output/study_1/drift_full_model_pr.nc')
print("Over")

In [ ]:
# Fitting the dataset simulated using the True Estimates (Study 2) 

pr_df_study2 = pd.read_csv('../task_data/study_2/drift_full_model_pr_true_param.csv')
pr_df_study2['rt'] = pr_df_study2['rt']/1000

model_b = hssm.HSSM(
    model = 'ddm',
    data = pr_df_study2,
    include = [
        {
            "name": "v",
            "formula": "v ~ 1 + r_diff_norm + delta_valence + (1 + r_diff_norm + delta_valence|subject)",       
        }
    ],
    
)

ddm_samples_b = model_b.sample(sampler = "mcmc", tune = 2000, draws = 10000, chains = 4, cores = 4,target_accept=0.95)
ddm_samples_b.to_netcdf('../../model_output/study_2/drift_full_model_pr.nc')
print("Over")